# Setup

## Import

In [1]:
import json
import logging
from pathlib import Path
from torch.utils.data import DataLoader
from datasets import load_from_disk
from model_testing.model import build_model_and_transforms, get_device, CollateFn
from model_testing.utils import run_inference, compute_metrics

## Costanti

In [2]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
# Constants
IMAGENET_1K_DATASET_PATH = Path("../preprocessed_datasets/imagenet_1k_preprocessed")
IMAGENET_R_DATASET_PATH = Path("../preprocessed_datasets/imagenet_r_preprocessed")

RESULTS_BASELINE_PATH = Path("../results/ImageNetR/baseline_metrics.json")
RESULTS_IMAGENET_R_PATH = Path("../results/ImageNetR/drift_metrics.json")

BATCH_SIZE = 64
NUM_WORKERS = 4
TOP_K = 5

# Test

## Setup modello e dataset

### Load device

In [3]:
device = get_device()
logger.info(f"Utilizing device: {device}")

INFO:__main__:Utilizing device: cuda


### Load dataset

In [4]:
if not IMAGENET_1K_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed dataset not found in {IMAGENET_1K_DATASET_PATH}. "
        "Please, execute the preprocessing script first."
    )
if not IMAGENET_R_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed dataset not found in {IMAGENET_R_DATASET_PATH}. "
        "Please, execute the preprocessing script first."
    )

logger.info("Loading preprocessed dataset...")
dataset_base = load_from_disk(str(IMAGENET_1K_DATASET_PATH))
dataset_r = load_from_disk(str(IMAGENET_R_DATASET_PATH))


INFO:__main__:Loading preprocessed dataset...


### Load model

In [5]:
logger.info("Loading ResNet50...")
model, preprocess = build_model_and_transforms()
model.to(device)

dataloader_base = DataLoader(
    dataset_base,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=CollateFn(preprocess=preprocess),
)
dataloader_r = DataLoader(
    dataset_r,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=CollateFn(preprocess=preprocess),
)


INFO:__main__:Loading ResNet50...


## Inferenza

In [6]:
y_true_base, y_pred_base, topk_correct_base, features_base = run_inference(
    model, dataloader_base, device, k=TOP_K, return_features=True
)
y_true_r, y_pred_r, topk_correct_r, features_r = run_inference(
    model, dataloader_r, device, k=TOP_K, return_features=True)

Model inference: 100%|██████████| 157/157 [00:21<00:00,  7.47it/s]


## Calcolo metriche

In [7]:
metrics_base = compute_metrics(y_true_base, y_pred_base, topk_correct_base)
metrics_r = compute_metrics(y_true_r, y_pred_r, topk_correct_r)

/home/daniele/magistrale/DataDrift-Project/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/daniele/magistrale/DataDrift-Project/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


### Stampa e salvataggio metriche

In [8]:
from model_testing.utils import save_features_csv

logger.info("Baseline metrics computed:")
for key, value in metrics_base.items():
    logger.info(f"  {key}: {value}")

with open(RESULTS_BASELINE_PATH, "w") as f:
    json.dump(metrics_base, f, indent=2)
logger.info(f"Metrics saved in {RESULTS_BASELINE_PATH}")
save_features_csv(features_base, y_true_base, y_pred_base, "../results/ImageNetR/baseline_features.csv")

INFO:__main__:Baseline metrics computed:
INFO:__main__:  accuracy: 0.8546
INFO:__main__:  top5_accuracy: 0.9668
INFO:__main__:  balanced_accuracy: 0.8546
INFO:__main__:  precision_macro: 0.2866908368512706
INFO:__main__:  recall_macro: 0.252094395280236
INFO:__main__:  f1_macro: 0.2665751213134408
INFO:__main__:  precision_weighted: 0.9718819369258075
INFO:__main__:  recall_weighted: 0.8546
INFO:__main__:  f1_weighted: 0.9036896612525644
INFO:__main__:  precision_micro: 0.8546
INFO:__main__:  recall_micro: 0.8546
INFO:__main__:  f1_micro: 0.8546
INFO:__main__:  mcc: 0.854055200411218
INFO:__main__:  cohen_kappa: 0.8539574859594787
INFO:__main__:  n_samples: 10000
INFO:__main__:  n_classes_present: 200
INFO:__main__:  min_samples_per_class: 50
INFO:__main__:  max_samples_per_class: 50
INFO:__main__:  mean_samples_per_class: 50.0
INFO:__main__:  std_samples_per_class: 0.0
INFO:__main__:Metrics saved in ../results/ImageNetR/baseline_metrics.json


In [9]:
logger.info("Drift metrics computed:")
for key, value in metrics_r.items():
    logger.info(f"  {key}: {value}")
with open(RESULTS_IMAGENET_R_PATH, "w") as f:
    json.dump(metrics_r, f, indent=2)
logger.info(f"Metrics saved in {RESULTS_IMAGENET_R_PATH}")
save_features_csv(features_r, y_true_r, y_pred_r, "../results/ImageNetR/drift_features.csv")


INFO:__main__:Drift metrics computed:
INFO:__main__:  accuracy: 0.2979
INFO:__main__:  top5_accuracy: 0.4494
INFO:__main__:  balanced_accuracy: 0.2979
INFO:__main__:  precision_macro: 0.1975766192125109
INFO:__main__:  recall_macro: 0.07178313253012047
INFO:__main__:  f1_macro: 0.09935753072288736
INFO:__main__:  precision_weighted: 0.8199429697319202
INFO:__main__:  recall_weighted: 0.2979
INFO:__main__:  f1_weighted: 0.4123337524999826
INFO:__main__:  precision_micro: 0.2979
INFO:__main__:  recall_micro: 0.2979
INFO:__main__:  f1_micro: 0.2979
INFO:__main__:  mcc: 0.2982102531252596
INFO:__main__:  cohen_kappa: 0.29659130812877366
INFO:__main__:  n_samples: 10000
INFO:__main__:  n_classes_present: 200
INFO:__main__:  min_samples_per_class: 50
INFO:__main__:  max_samples_per_class: 50
INFO:__main__:  mean_samples_per_class: 50.0
INFO:__main__:  std_samples_per_class: 0.0
INFO:__main__:Metrics saved in ../results/ImageNetR/drift_metrics.json
